In [ ]:
#############################
### Overview of the code: ###
#############################

# Image classification using Hugging Face Transformers
# First we will build MNIST image classification using PyTorch. 
# This will explain the basics of using Hugging Face Transformers for image classification.
# Later we will expand this to more complex datasets and models.
# I am planning to use the CFIAR-10 dataset for image classification. 
# It has 10 classes of images and is a common benchmark for image classification tasks.
# We will build basic logistic regression model using PyTorch for CFIAR-10 dataset.
# We will show challenges of building image classification models using Logistic Regression.
# We will display learning curves for Logistic Regression model. 
# And accuracy on test set will be calculated.
# After that we will build a simple Deeplearning model for image classification using PyTorch.
# we wills show differences between Logistic Regression and Deep Learning models for image classification.
# We will also show learning curves and test accuracy for Deep Learning model.
# And then we will build CNN model for image classification using PyTorch.
# We will show how CNNs are better than Logistic Regression and Deep Learning models for image classification
# We will also show learning curves and test accuracy for CNN model.
# Show the differences in performance between Logistic Regression, Deep Learning and 
# CNN models for image classification.
# And the finally we will build a ResNet model for image classification using PyTorch.
# We will show how ResNet is better than Logistic Regression, Deep Learning, 
# CNN and resnet models for image classification.
# We will also show learning curves and test accuracy for ResNet model.

# Finally we will data augmentation and regularization techniques for image classification using PyTorch.
# We will show how data augmentation and regularization can improve performance of image classification models.
# We will perform data augmentation using techniques like random cropping, flipping, and rotation.
# We will also show how regularization techniques like dropout and weight decay 
# can improve performance of image classification models.
# After applying data augmentation and regularization techniques we will 
# re-evaluate the performance of all the models and show the improvements in learning curves and test accuracy.

# Let us build code in modular way so that we can easily reuse the code for different models and datasets. 
# Also we re-evaluate the performance of all the models after applying data augmentation and 
# regularization techniques.

In [ ]:
# Let us start with MNIST image classification using Logistic Regression model.
# import necessary libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torch.utils.tensorboard import SummaryWriter
from torchvision.datasets import MNIST
from matplotlib import pyplot as plt
import numpy as np


In [ ]:
# define the device to be used for training
def device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# let us define the parameters for training
def model_parameters():
    params = {
        "batch_size": 64,
        "learning_rate": 0.01,
        "num_epochs": 10,
        "input_size": 28 * 28,  # MNIST images are 28x28
        "num_classes": 10,       # 10 classes for MNIST
    }
    return params

In [ ]:
# Load the MNIST dataset both training and test sets
def load_mnist_data(params):
    train_dataset = MNIST(root='./data', train=True, download=True, transform=transforms.ToTensor())
    test_dataset = MNIST(root='./data', train=False, download=True, transform=transforms.ToTensor())
    train_loader = DataLoader(train_dataset, batch_size=params["batch_size"], shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=params["batch_size"], shuffle=False)
    return train_loader, test_loader

In [ ]:
# split the train dataset into train and validation sets
def split_train_val(train_loader, params, val_split=0.2):
    train_size = int(len(train_loader.dataset) * (1 - val_split))
    val_size = len(train_loader.dataset) - train_size
    train_dataset, val_dataset = torch.utils.data.random_split(train_loader.dataset, [train_size, val_size])
    train_loader = DataLoader(train_dataset, batch_size=params["batch_size"], shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=params["batch_size"], shuffle=False)
    return train_loader, val_loader

In [ ]:
# Show the sample images from the dataset and size of the dataset
def show_sample_images(train_loader):
    for images, labels in train_loader:
        fig, axes = plt.subplots(1, 5, figsize=(10, 2))
        for i in range(5):
            axes[i].imshow(images[i].squeeze(), cmap='gray')
            axes[i].set_title(f'Label: {labels[i].item()}')
            axes[i].axis('off')
        plt.show()
        break
    print(f"Number of training examples: {len(train_loader.dataset)}")
    return

In [ ]:
# calculate the accuracy of the model on valudation and test sets. Also calculate loss on test set
def calculate_metrics(model, val_loader, test_loader, criterion, device):
    model.to(device)
    with torch.no_grad():
        model.eval()
        val_correct = 0
        val_total = 0
        val_loss = 0
        for images, labels in val_loader:
            images = images.view(-1, 28 * 28)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
        val_accuracy = 100 * val_correct / val_total

        test_correct = 0
        test_total = 0
        test_loss = 0
        for images, labels in test_loader:
            images = images.view(-1, 28 * 28)
            outputs = model(images)
            loss = criterion(outputs, labels)
            test_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            test_total += labels.size(0)
            test_correct += (predicted == labels).sum().item()
        test_accuracy = 100 * test_correct / test_total
    return val_accuracy, test_accuracy, val_loss, test_loss

In [ ]:
# display the learning curves for training and validation loss and accuracy
def display_learning_curves(train_losses, val_losses, train_accuracies, val_accuracies):
    epochs = range(1, len(train_losses) + 1)
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(epochs, train_losses, label='Training Loss')
    plt.plot(epochs, val_losses, label='Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.title('Loss Curves')
    plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(epochs, train_accuracies, label='Training Accuracy')
    plt.plot(epochs, val_accuracies, label='Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy (%)')
    plt.title('Accuracy Curves')
    plt.legend()
    plt.show()

In [ ]:
# define the training loop for the model
def train_model(model, train_loader, val_loader, test_loader, params, device):
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=params["learning_rate"], momentum=0.9)
    train_losses = []
    val_losses = []
    test_losses = []    
    train_accuracies = []
    val_accuracies = []
    test_accuracies = []
    for epoch in range(params["num_epochs"]):
        model.train()
        for images, labels in train_loader:
            images = images.view(-1, 28 * 28)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
        val_accuracy, test_accuracy, val_loss, test_loss = calculate_metrics(model, val_loader, test_loader, criterion, device())
        train_losses.append(loss.item())
        val_losses.append(val_loss)
        test_losses.append(test_loss)
        train_accuracies.append(val_accuracy)
        val_accuracies.append(test_accuracy)
        test_accuracies.append(test_accuracy)
        print(f"Epoch [{epoch + 1}/{params['num_epochs']}], Loss: {loss.item()}, Val Accuracy: {val_accuracy:.2f}, Test Accuracy: {test_accuracy:.2f}")
    return model, train_losses, val_losses, test_losses, train_accuracies, val_accuracies

In [ ]:
# define the function to check predict image and label from the test set
def predict_image(model, test_loader, device):
    model.to(device)
    model.eval()
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.view(-1, 28 * 28)
            images = images.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            print(f"Predicted label: {predicted.item()}, Actual label: {labels.item()}")
            break
    return
    

In [ ]:
# Now we have all modules defined for training the model, calculating metrics and displaying learning curves.
# Let us now build the Logistic Regression model for MNIST image classification.

# define the Logistic Regression model for MNIST image classification
class LogisticRegression(nn.Module):
    def __init__(self, input_size, num_classes):
        super(LogisticRegression, self).__init__()
        self.linear = nn.Linear(input_size, num_classes)

    def forward(self, x):
        out = self.linear(x)
        return out
    

In [ ]:
# create an instance of the Logistic Regression model
def create_logistic_regression_model(params):
    model = LogisticRegression(params["input_size"], params["num_classes"])
    return model

In [ ]:
device = device()
print(f"Using device: {device}")

model = create_logistic_regression_model(model_parameters())
train_loader, test_loader = load_mnist_data(model_parameters())
train_loader, val_loader = split_train_val(train_loader, model_parameters())
show_sample_images(train_loader)
model, train_losses, val_losses, test_losses, train_accuracies, val_accuracies = train_model(model, train_loader, val_loader, test_loader, model_parameters(), device())
display_learning_curves(train_losses, val_losses, train_accuracies, val_accuracies)
predict_image(model, test_loader, device())